# YouTube オーディエンス・ネットワーク分析

「視聴者が他にどんなチャンネルを見ているか」を、視聴者 × チャンネルの2部グラフとして分析するプロジェクト。

**スコープ（合意事項）**
- 出力は**集計のみ**。個人単位の登録チャンネル一覧は計算後に保持しない。
- 特定個人の名指し・プロファイリングはしない。
- 視聴者像はジャンル・系統の傾向という粗い粒度に留める。
- コメント投稿者のIDは計算の中間データとしてのみメモリ上で扱い、Stage 4 で匿名化する。

**進捗**
- [x] Stage 1: Google Cloud で API キー取得
- [x] Stage 2: 環境構築・接続テスト
- [ ] Stage 3: データ収集（コメント投稿者 → 公開 subscriptions） ← いまここ
- [ ] Stage 4: ネットワーク構築（匿名化した2部グラフ）
- [ ] Stage 5: 分析（共起チャンネル集計など）
- [ ] Stage 6: 可視化


## 共通設定

API キーは Codespaces secret（環境変数 `YOUTUBE_API_KEY`）から読み込む。ローカルなら `.env` でも同じコードで動く。

In [1]:
import os
from dotenv import load_dotenv
from googleapiclient.discovery import build

load_dotenv()  # ローカルなら .env を読む。Codespaces secret なら何もしない

API_KEY = os.environ.get("YOUTUBE_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "YOUTUBE_API_KEY が見つかりません。Codespaces secret か .env を確認してください。"
    )

youtube = build("youtube", "v3", developerKey=API_KEY)
print("クライアント作成 OK")

クライアント作成 OK


In [2]:
# 分析対象チャンネル（オーナー権限あり）。Stage 2 で確認済みのIDを使用。
CHANNEL_ID = "UCYo5jkYvVZY7wGItoRELILg"

# 接続テスト（公開情報の取得確認）
resp = youtube.channels().list(part="snippet,statistics", id=CHANNEL_ID).execute()
ch = resp["items"][0]
print("チャンネル名 :", ch["snippet"]["title"])
print("登録者数     :", ch["statistics"].get("subscriberCount", "非公開"))
print("動画数       :", ch["statistics"].get("videoCount"))

チャンネル名 : 内田博史【金持ちの習慣】
登録者数     : 213000
動画数       : 659


## Stage 3: データ収集

**手順**
1. 対象チャンネルのアップロード動画一覧を取得
2. 各動画のコメントから「コメント投稿者（＝視聴者の代理）」を集める
3. 各投稿者の**公開**登録チャンネルを取得し、`(視聴者, チャンネル)` のエッジを作る

**まずは小さくテスト**するためのパラメータが下のセル。`no_public_subs`（公開subが取れなかった人数）が多いのは想定どおり。取得できたエッジ数を見てから本番規模を決める。

In [ ]:
# --- テスト用パラメータ（最初はこの規模で動作確認）---
MAX_VIDEOS = 659                 # コメントを集める動画数（最新からN本）
MAX_COMMENTS_PER_VIDEO = 600   # 1動画あたり最大コメント数（100 = 1ページ）
MAX_SUBS_PER_USER = 1000         # 1ユーザーあたり取得する登録chの上限（50 = 1ページ）

In [4]:
# 1) アップロード動画の一覧から最新 MAX_VIDEOS 本の videoId を取得
ch_resp = youtube.channels().list(part="contentDetails", id=CHANNEL_ID).execute()
uploads_playlist = ch_resp["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]

video_ids = []
req = youtube.playlistItems().list(
    part="contentDetails", playlistId=uploads_playlist, maxResults=50
)
while req is not None and len(video_ids) < MAX_VIDEOS:
    resp = req.execute()
    for item in resp["items"]:
        video_ids.append(item["contentDetails"]["videoId"])
        if len(video_ids) >= MAX_VIDEOS:
            break
    req = youtube.playlistItems().list_next(req, resp)

video_ids = video_ids[:MAX_VIDEOS]
print(f"対象動画: {len(video_ids)} 本")

対象動画: 423 本


In [6]:
# 2) 各動画のコメント投稿者を集める
from googleapiclient.errors import HttpError

commenter_ids = set()
skipped = 0
for vid in video_ids:
    collected = 0
    req = youtube.commentThreads().list(
        part="snippet", videoId=vid, maxResults=100, textFormat="plainText"
    )
    while req is not None and collected < MAX_COMMENTS_PER_VIDEO:
        try:
            resp = req.execute()
        except HttpError:
            # コメント無効・非公開などの動画はスキップ
            print(f"動画 {vid} はスキップ（コメント無効など）")
            skipped += 1
            break
        for item in resp["items"]:
            top = item["snippet"]["topLevelComment"]["snippet"]
            author = top.get("authorChannelId", {}).get("value")
            if author and author != CHANNEL_ID:   # チャンネル主自身は除く
                commenter_ids.add(author)
            collected += 1
            if collected >= MAX_COMMENTS_PER_VIDEO:
                break
        req = youtube.commentThreads().list_next(req, resp)

print(f"ユニークなコメント投稿者: {len(commenter_ids)} 人（スキップ動画: {skipped} 本）")

動画 lxpj9PyNV0A はスキップ（コメント無効など）
ユニークなコメント投稿者: 9497 人（スキップ動画: 1 本）


In [7]:
# 3) 各投稿者の公開登録チャンネルを取得し、エッジを作る
#    edges は中間データ（メモリ上のみ）。Stage 4 で匿名化する。
from tqdm.auto import tqdm

edges = []          # (commenter_id, subscribed_channel_id, subscribed_channel_title)
no_public_subs = 0  # 登録が非公開 or 取得不可だった人数

for uid in tqdm(commenter_ids):
    try:
        resp = youtube.subscriptions().list(
            part="snippet", channelId=uid, maxResults=MAX_SUBS_PER_USER
        ).execute()
    except Exception:
        no_public_subs += 1          # 多くは「登録を非公開」にしている人
        continue
    items = resp.get("items", [])
    if not items:
        no_public_subs += 1
        continue
    for it in items:
        sub_id = it["snippet"]["resourceId"]["channelId"]
        sub_title = it["snippet"]["title"]
        edges.append((uid, sub_id, sub_title))

print(f"取得できたエッジ数      : {len(edges)}")
print(f"公開subが取れた人        : {len(commenter_ids) - no_public_subs} / {len(commenter_ids)}")
print(f"公開subが取れなかった人  : {no_public_subs}")

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 9497/9497 [06:07<00:00, 25.84it/s]

取得できたエッジ数      : 64422
公開subが取れた人        : 1497 / 9497
公開subが取れなかった人  : 8000


In [8]:
# 4) 集計プレビュー：視聴者層に多く共有されているチャンネル（＝Stage 5 の核の先取り）
import pandas as pd

df = pd.DataFrame(edges, columns=["commenter", "channel_id", "channel_title"])

# 自分自身のチャンネルは除外（当然みんな登録しているため）
df = df[df["channel_id"] != CHANNEL_ID]

# 1人が同じchを二重に数えないよう重複を除いてから、登録者数をカウント
unique_pairs = df.drop_duplicates(subset=["commenter", "channel_id"])
top = (unique_pairs.groupby(["channel_id", "channel_title"])["commenter"]
       .nunique().sort_values(ascending=False).head(20))

print("=== 視聴者が多く登録している他チャンネル Top20 ===")
print(top)

=== 視聴者が多く登録している他チャンネル Top20 ===
channel_id                channel_title                         
UC0PotgTwYxbOZuM9FbWKdcg  Paranoia_パラノイア【有益】                        460
UC-N7pA0rR1PrUBOfs2OA0oA  哲理学作家さとうみつろう『神さまとのおしゃべり』チャンネル             320
UC67Wr_9pA4I0glIxDt_Cpyw  両学長 リベラルアーツ大学                             267
UC4lN5sizuJraSHqy99xTy6Q  Naokiman Show                             205
UC1WkFVOCTPdY782AJ1PZ-JQ  Dr. Zion Kabasawa                         170
UC0FFHRF1mytLDhs6nxqGIQg  OTAKING / Toshio Okada                    152
UC8yHePe_RgUBE-waRWy6olw  PIVOT 公式チャンネル                             144
UC1bmnUH1ffc63zohkPvtXcQ  ブライトサイド | Bright Side Japan               133
UCFo4kqllbcQ4nV83WCyraiw  NAKATA UNIVERSITY                         123
UC-kF1uMFhIfvw6seHqDGwkg  アシタノワダイ                                   119
UC0yQ2h4gQXmVUFWZSqlMVOA  ひろゆき, hiroyuki                            111
UC9V4eJBNx_hOieGG51NZ6nA  フェルミ漫画大学                                  109
UCEixleMT76xDzoiEb9ZA7XA  本要約チャンネル【毎日1

### 結果の見方 / 次の判断

- `no_public_subs` が多くても正常（登録を非公開にしている人は取れない）。
- 取得できたエッジ数が少なすぎる場合は、`MAX_VIDEOS` と `MAX_COMMENTS_PER_VIDEO` を増やして視聴者の母数を広げる。
- このプレビューの Top チャンネルが「自分の視聴者が他に見ているチャンネル」の素データ。

ここまでの数字（コメント投稿者数 / 公開subが取れた人数 / エッジ数 / Top20）を確認できたら、本番規模を決めて **Stage 4（匿名化した2部グラフの構築）** に進む。

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from tqdm.auto import tqdm

# ---- セットアップ（このセル内で完結）----
load_dotenv()
API_KEY = os.environ.get("YOUTUBE_API_KEY")
if not API_KEY:
    raise RuntimeError("YOUTUBE_API_KEY が見つかりません（Codespaces secret / .env を確認）")
youtube = build("youtube", "v3", developerKey=API_KEY)

CHANNEL_ID = "UCYo5jkYvVZY7wGItoRELILg"
CACHE = "edges_anon.csv"

MAX_VIDEOS = 659                # ← 前回の値に合わせる
MAX_COMMENTS_PER_VIDEO = 600    # ← 前回の値に合わせる
MAX_SUBS_PER_USER = 300

# ---- 読み込み or 収集 ----
if os.path.exists(CACHE):
    df_anon = pd.read_csv(CACHE)
    print("キャッシュから読み込み（収集スキップ）:", df_anon.shape)
else:
    print("キャッシュなし → 収集を実行します（クォータ消費）")

    # 1) アップロード動画の最新 MAX_VIDEOS 本
    ch = youtube.channels().list(part="contentDetails", id=CHANNEL_ID).execute()
    uploads = ch["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]
    video_ids = []
    req = youtube.playlistItems().list(part="contentDetails", playlistId=uploads, maxResults=50)
    while req is not None and len(video_ids) < MAX_VIDEOS:
        r = req.execute()
        video_ids += [i["contentDetails"]["videoId"] for i in r["items"]]
        req = youtube.playlistItems().list_next(req, r)
    video_ids = video_ids[:MAX_VIDEOS]
    print("対象動画:", len(video_ids))

    # 2) コメント投稿者（コメント無効動画はスキップ）
    commenter_ids = set()
    for vid in tqdm(video_ids, desc="comments"):
        collected = 0
        req = youtube.commentThreads().list(part="snippet", videoId=vid,
                                            maxResults=100, textFormat="plainText")
        while req is not None and collected < MAX_COMMENTS_PER_VIDEO:
            try:
                r = req.execute()
            except HttpError:
                break
            for it in r["items"]:
                a = it["snippet"]["topLevelComment"]["snippet"].get("authorChannelId", {}).get("value")
                if a and a != CHANNEL_ID:
                    commenter_ids.add(a)
                collected += 1
                if collected >= MAX_COMMENTS_PER_VIDEO:
                    break
            req = youtube.commentThreads().list_next(req, r)
    print("コメント投稿者:", len(commenter_ids))

    # 3) 公開subscriptions
    rows, no_pub = [], 0
    for uid in tqdm(commenter_ids, desc="subscriptions"):
        try:
            r = youtube.subscriptions().list(part="snippet", channelId=uid,
                                             maxResults=MAX_SUBS_PER_USER).execute()
        except Exception:
            no_pub += 1
            continue
        items = r.get("items", [])
        if not items:
            no_pub += 1
            continue
        for it in items:
            rows.append((uid, it["snippet"]["resourceId"]["channelId"], it["snippet"]["title"]))
    print(f"公開subが取れた人: {len(commenter_ids)-no_pub}/{len(commenter_ids)}  エッジ: {len(rows)}")

    df = pd.DataFrame(rows, columns=["commenter", "channel_id", "channel_title"])

    # 4) 匿名化（対応表は即破棄）→ キャッシュ保存
    amap = {u: f"u{i}" for i, u in enumerate(df["commenter"].unique())}
    df_anon = df.copy()
    df_anon["viewer"] = df_anon["commenter"].map(amap)
    df_anon = df_anon.drop(columns=["commenter"])
    del amap
    df_anon = (df_anon[df_anon["channel_id"] != CHANNEL_ID]
               .drop_duplicates(subset=["viewer", "channel_id"]))
    df_anon.to_csv(CACHE, index=False)
    print("匿名エッジを保存:", df_anon.shape)

キャッシュから読み込み（収集スキップ）: (65538, 3)


In [2]:
# Stage 4: ノイズ除去と分析対象の確定
# コメント投稿者かつ公開登録チャンネルを取得できた人を「観測パネル」と呼ぶ。
# これはチャンネル全体の視聴者母集団ではないため、以後の比率もこのパネル内の値として解釈する。

import numpy as np
import pandas as pd

N_OBSERVED_PANEL = df_anon["viewer"].nunique()

# 1人だけが登録しているチャンネルは、共起ネットワークの根拠として弱いため除外する。
MIN_VIEWERS = 3

# 残存チャンネルが1件だけの視聴者は、チャンネル間の共起を作れないため除外する。
MIN_CHANNELS_PER_VIEWER = 2


def first_nonempty(values):
    """チャンネル名の表記揺れに対し、最初の非空値を採用する。"""
    values = values.dropna()
    return values.iloc[0] if not values.empty else None


def summarize_channel_population(edges):
    """channel_id 単位で、観測パネル内の登録者数を集計する。"""
    return (
        edges.groupby("channel_id")
        .agg(
            channel_title=("channel_title", first_nonempty),
            n_viewers=("viewer", "nunique"),
        )
        .reset_index()
        .sort_values("n_viewers", ascending=False)
    )


# チャンネルと視聴者を交互に除外し、閾値を満たす部分グラフに収束させる。
# これにより、最終的な channel_pop と edges_f の母集団を一致させる。
edges_f = (
    df_anon[["viewer", "channel_id", "channel_title"]]
    .drop_duplicates(subset=["viewer", "channel_id"])
    .copy()
)

for _ in range(10):
    before_shape = edges_f.shape

    population = summarize_channel_population(edges_f)
    kept = set(
        population.loc[
            population["n_viewers"] >= MIN_VIEWERS,
            "channel_id",
        ]
    )
    edges_f = edges_f[edges_f["channel_id"].isin(kept)].copy()

    viewer_degree = (
        edges_f.groupby("viewer")["channel_id"]
        .nunique()
    )
    valid_viewers = set(
        viewer_degree[
            viewer_degree >= MIN_CHANNELS_PER_VIEWER
        ].index
    )
    edges_f = edges_f[edges_f["viewer"].isin(valid_viewers)].copy()

    if edges_f.shape == before_shape:
        break

channel_pop_all = summarize_channel_population(
    df_anon[["viewer", "channel_id", "channel_title"]]
    .drop_duplicates(subset=["viewer", "channel_id"])
)

channel_pop = summarize_channel_population(edges_f)
kept = set(channel_pop["channel_id"])

title_map = (
    channel_pop_all
    .set_index("channel_id")["channel_title"]
    .to_dict()
)

print(f"観測パネル人数: {N_OBSERVED_PANEL:,}")
print(f"全チャンネル数: {channel_pop_all.shape[0]:,}")
print(
    f"最終ネットワーク対象チャンネル数 "
    f"(登録者 {MIN_VIEWERS} 人以上): {len(kept):,}"
)
print(
    f"最終ネットワーク対象視聴者数 "
    f"(登録チャンネル {MIN_CHANNELS_PER_VIEWER} 件以上): "
    f"{edges_f['viewer'].nunique():,}"
)
print(f"最終ネットワーク用エッジ数: {len(edges_f):,}")


観測パネル人数: 1,527
全チャンネル数: 31,277
最終ネットワーク対象チャンネル数 (登録者 3 人以上): 4,217
最終ネットワーク対象視聴者数 (登録チャンネル 2 件以上): 1,497
最終ネットワーク用エッジ数: 34,866


In [4]:
# Stage 5: 登録チャンネル共起ネットワークの構築
# 単純な二部グラフ投影では、登録チャンネル数が多い視聴者1人が大量の結線を作る。
# そこで、各視聴者の寄与を 1 / (登録チャンネル数 - 1) に近づける重み付けを行う。

import networkx as nx
from scipy.sparse import csr_matrix, triu

MIN_SHARED_VIEWERS = 3
MIN_WEIGHTED_COSINE = 0.12

if edges_f.empty:
    raise ValueError("ネットワーク用エッジがありません。MIN_VIEWERS を下げてください。")

viewer_index = pd.Index(
    edges_f["viewer"].unique(),
    name="viewer",
)
channel_index = pd.Index(
    sorted(edges_f["channel_id"].unique()),
    name="channel_id",
)

row = viewer_index.get_indexer(edges_f["viewer"])
col = channel_index.get_indexer(edges_f["channel_id"])

# 行 = 視聴者、列 = チャンネルの二値行列
X = csr_matrix(
    (
        np.ones(len(edges_f), dtype=np.float32),
        (row, col),
    ),
    shape=(len(viewer_index), len(channel_index)),
)
X.sum_duplicates()

viewer_degree = np.asarray(X.sum(axis=1)).ravel()
valid_rows = viewer_degree > 1

X = X[valid_rows]
viewer_degree = viewer_degree[valid_rows]

# 各視聴者が多くのチャンネルを登録しているほど、1組あたりの寄与を下げる。
# X_weighted.T @ X_weighted の各共起には 1 / (degree - 1) が加わる。
viewer_weight = 1 / np.sqrt(viewer_degree - 1)
X_weighted = X.multiply(viewer_weight[:, None]).tocsr()

# raw_cooccurrence: 生の共通登録者数
# weighted_cooccurrence: ヘビーユーザーの影響を補正した共起
raw_cooccurrence = (X.T @ X).tocsr()
weighted_cooccurrence = (X_weighted.T @ X_weighted).tocsr()

upper = triu(raw_cooccurrence, k=1, format="coo")
raw_shared = upper.data.astype(int)

weighted_overlap = np.asarray(
    weighted_cooccurrence[upper.row, upper.col]
).reshape(-1)

weighted_support = np.asarray(
    X_weighted.power(2).sum(axis=0)
).reshape(-1)

denominator = np.sqrt(
    weighted_support[upper.row]
    * weighted_support[upper.col]
)

weighted_cosine = np.divide(
    weighted_overlap,
    denominator,
    out=np.zeros_like(weighted_overlap, dtype=float),
    where=denominator > 0,
)

# 「共通登録者が少ない偶然の接続」と「正規化しても弱い接続」を除く。
mask = (
    (raw_shared >= MIN_SHARED_VIEWERS)
    & (weighted_cosine >= MIN_WEIGHTED_COSINE)
)

edge_table = pd.DataFrame({
    "channel_id_a": channel_index[upper.row[mask]],
    "channel_id_b": channel_index[upper.col[mask]],
    "shared_viewers": raw_shared[mask],
    "fractional_overlap": weighted_overlap[mask],
    "weighted_cosine": weighted_cosine[mask],
})

G_ch = nx.Graph()
G_ch.add_edges_from(
    (
        row.channel_id_a,
        row.channel_id_b,
        {
            "weight": float(row.weighted_cosine),
            "shared_viewers": int(row.shared_viewers),
            "fractional_overlap": float(row.fractional_overlap),
        },
    )
    for row in edge_table.itertuples(index=False)
)

print(f"正規化後ネットワークのノード数: {G_ch.number_of_nodes():,}")
print(f"正規化後ネットワークのエッジ数: {G_ch.number_of_edges():,}")
print(f"共通登録者数の下限: {MIN_SHARED_VIEWERS}")
print(f"正規化コサイン類似度の下限: {MIN_WEIGHTED_COSINE}")


正規化後ネットワークのノード数: 1,679
正規化後ネットワークのエッジ数: 4,023
共通登録者数の下限: 3
正規化コサイン類似度の下限: 0.12


In [5]:
# Stage 5-補助: 正規化後ネットワークの中心チャンネルを確認
# ここでの重み付き次数は、単純な共通登録者数ではなく
# ヘビーユーザーの影響を補正したコサイン類似度の合計である。

if G_ch.number_of_nodes() == 0:
    raise ValueError(
        "フィルタ後のエッジがありません。"
        "MIN_SHARED_VIEWERS または MIN_WEIGHTED_COSINE を下げてください。"
    )

wdeg = sorted(
    G_ch.degree(weight="weight"),
    key=lambda item: item[1],
    reverse=True,
)[:15]

print("=== 正規化後ネットワークの中心チャンネル（重み付き次数 上位）===")
for channel_id, weighted_degree in wdeg:
    print(
        f"{title_map.get(channel_id, channel_id)[:34]:34s} "
        f"{weighted_degree:.4f}"
    )

edge_table.to_csv(
    "channel_network_edges_filtered.csv",
    index=False,
)


=== 正規化後ネットワークの中心チャンネル（重み付き次数 上位）===
哲理学作家さとうみつろう『神さまとのおしゃべり』チャンネル      18.3983
Paranoia_パラノイア【有益】                 11.6581
アシタノワダイ                            10.4838
OTAKING / Toshio Okada             8.3732
ブライトサイド | Bright Side Japan        8.3317
宇宙となかよし/Qさん                        7.3451
Nakano Hiroshi BooKtube Univ.      7.1176
IROHA TAROT                        7.0712
【FX初心者ch】 by ユーちぇる監督               6.8411
桜井美帆のオーラで開運チャンネル                   6.6499
なるためJAPAN                          6.6308
NAKATA UNIVERSITY                  6.5540
占い師けんけんTV                          6.4905
FXメガバンク – 今日から使えるトレード講座            6.4765
神結ちゃんねる 〜かみすちゃんねる〜                 6.4711


In [6]:
# Stage 6: Louvainクラスタリング
# クラスタ代表は「観測パネル内で単に人気」なチャンネルではなく、
# クラスタ内部での正規化された結びつきが強いチャンネルとして選ぶ。

from networkx.algorithms.community import louvain_communities

LOUVAIN_RESOLUTION = 1.0

communities = louvain_communities(
    G_ch,
    weight="weight",
    resolution=LOUVAIN_RESOLUTION,
    seed=42,
)

communities = sorted(
    communities,
    key=lambda comm: (-len(comm), sorted(comm)[0]),
)

pop = channel_pop.set_index("channel_id")

community_rows = []
channel_community_rows = []

for community_id, comm in enumerate(communities):
    subgraph = G_ch.subgraph(comm)
    internal_strength = dict(
        subgraph.degree(weight="weight")
    )

    representatives = sorted(
        comm,
        key=lambda channel_id: (
            internal_strength.get(channel_id, 0),
            pop["n_viewers"].get(channel_id, 0),
        ),
        reverse=True,
    )[:8]

    community_rows.append({
        "community_id": community_id,
        "n_channels": len(comm),
        "n_edges": subgraph.number_of_edges(),
        "internal_weight": round(
            sum(
                data["weight"]
                for _, _, data in subgraph.edges(data=True)
            ),
            4,
        ),
        "representative_channels": " / ".join(
            title_map.get(channel_id, channel_id)
            for channel_id in representatives
        ),
    })

    for channel_id in comm:
        channel_community_rows.append({
            "channel_id": channel_id,
            "channel": title_map.get(channel_id, channel_id),
            "community_id": community_id,
            "sample_viewers": int(
                pop["n_viewers"].get(channel_id, 0)
            ),
            "internal_strength": round(
                internal_strength.get(channel_id, 0),
                6,
            ),
        })

community_summary = (
    pd.DataFrame(community_rows)
    .sort_values(
        ["n_channels", "internal_weight"],
        ascending=False,
    )
)

channel_communities = (
    pd.DataFrame(channel_community_rows)
    .sort_values(
        ["community_id", "internal_strength"],
        ascending=[True, False],
    )
)

print(
    f"コミュニティ数: {len(communities)} "
    f"(resolution={LOUVAIN_RESOLUTION})"
)
print()
print(
    community_summary.head(20).to_string(
        index=False,
        max_colwidth=100,
    )
)

community_summary.to_csv(
    "channel_community_summary.csv",
    index=False,
)

channel_communities.to_csv(
    "channel_communities.csv",
    index=False,
)


コミュニティ数: 55 (resolution=1.0)

 community_id  n_channels  n_edges  internal_weight                                                                              representative_channels
            0         272      488          84.6540 NAKATA UNIVERSITY  / 両学長 リベラルアーツ大学 / Paranoia_パラノイア【有益】 / PIVOT 公式チャンネル / 学識サロン / メンタリスト DaiGo / ...
            1         235      597          99.8361 哲理学作家さとうみつろう『神さまとのおしゃべり』チャンネル / 宇宙となかよし/Qさん / 桜井美帆のオーラで開運チャンネル / 【完全覚醒の学校】YUKARI / プロ霊能力者チャンネル / ...
            2         219      516          97.2697 アシタノワダイ / なるためJAPAN / Nakano Hiroshi BooKtube Univ. / 占い師けんけんTV / カピバラチャンネル capybarachannel / Hai...
            3         127      186          36.0901 井川意高が熔ける日本を斬る / OTAKING / Toshio Okada / シュン@億マーケ / ひろゆき, hiroyuki / ひろゆきの部屋【ひろゆき, hiroyuki】切り抜き ...
            4         107      158          33.6888 Naokiman Show / NMS STUDIO / Dharma Talk through Scary Stories / TOLAND VLOG / ウマヅラのお茶の間 / ねずみ / ...
            5          87      123          24.6049 

In [7]:
# Stage 7: 観測パネル内の重なりを算出
# これは「親和性」の確定値ではない。
# 比較対象チャンネル群がないため、ここでは
# 「コメント投稿者かつ公開登録取得者の観測パネル内での重なり」を出す。

from pathlib import Path
from datetime import datetime, timezone
from math import sqrt

MIN_OBSERVED_COMMENTERS = 5
STATS_CACHE_FILE = Path("channel_statistics_cache.csv")

# True にすると、キャッシュ済みのチャンネルも再取得する。
REFRESH_CHANNEL_STATS = False


def wilson_lower_bound(successes, total, z=1.96):
    """少数サンプルの偶然の上振れを抑える95% Wilson下限。"""
    if total == 0:
        return 0.0

    p = successes / total
    denominator = 1 + (z ** 2 / total)
    center = p + (z ** 2 / (2 * total))
    margin = z * sqrt(
        (p * (1 - p) / total)
        + (z ** 2 / (4 * total ** 2))
    )
    return max(0.0, (center - margin) / denominator)


candidate_pop = (
    channel_pop[
        channel_pop["n_viewers"] >= MIN_OBSERVED_COMMENTERS
    ]
    .copy()
)

channel_ids = candidate_pop["channel_id"].tolist()

# 取得済み統計は再利用し、不要な API コールを避ける。
if STATS_CACHE_FILE.exists():
    stats_cache = pd.read_csv(
        STATS_CACHE_FILE,
        dtype={"channel_id": str},
    )
else:
    stats_cache = pd.DataFrame(columns=[
        "channel_id",
        "total_subs",
        "subscriber_count_hidden",
        "retrieved_at",
    ])

if not stats_cache.empty:
    stats_cache["channel_id"] = (
        stats_cache["channel_id"]
        .astype(str)
    )

cached_ids = (
    set(stats_cache["channel_id"])
    if not stats_cache.empty
    else set()
)

ids_to_fetch = (
    channel_ids
    if REFRESH_CHANNEL_STATS
    else [
        channel_id
        for channel_id in channel_ids
        if channel_id not in cached_ids
    ]
)

new_rows = []

for i in range(0, len(ids_to_fetch), 50):
    batch = ids_to_fetch[i:i + 50]

    response = youtube.channels().list(
        part="statistics",
        id=",".join(batch),
    ).execute()

    returned_ids = set()

    for item in response.get("items", []):
        channel_id = item["id"]
        statistics = item.get("statistics", {})
        subscriber_count = statistics.get("subscriberCount")

        new_rows.append({
            "channel_id": channel_id,
            "total_subs": (
                int(subscriber_count)
                if subscriber_count is not None
                else None
            ),
            "subscriber_count_hidden": bool(
                statistics.get(
                    "hiddenSubscriberCount",
                    False,
                )
            ),
            "retrieved_at": datetime.now(
                timezone.utc
            ).isoformat(),
        })
        returned_ids.add(channel_id)

    # 応答に含まれないIDも記録し、次回も同じIDを無限に取りに行かない。
    for channel_id in set(batch) - returned_ids:
        new_rows.append({
            "channel_id": channel_id,
            "total_subs": None,
            "subscriber_count_hidden": None,
            "retrieved_at": datetime.now(
                timezone.utc
            ).isoformat(),
        })

if new_rows:
    stats_cache = pd.concat(
        [stats_cache, pd.DataFrame(new_rows)],
        ignore_index=True,
    )
    stats_cache = (
        stats_cache
        .drop_duplicates(
            subset=["channel_id"],
            keep="last",
        )
    )

stats_cache.to_csv(
    STATS_CACHE_FILE,
    index=False,
)

panel_overlap_df = candidate_pop.merge(
    stats_cache[
        [
            "channel_id",
            "total_subs",
            "subscriber_count_hidden",
            "retrieved_at",
        ]
    ],
    on="channel_id",
    how="left",
)

panel_overlap_df = panel_overlap_df.rename(columns={
    "channel_title": "channel",
    "n_viewers": "observed_commenters",
})

panel_overlap_df["observed_panel_size"] = N_OBSERVED_PANEL
panel_overlap_df["sample_share"] = (
    panel_overlap_df["observed_commenters"]
    / panel_overlap_df["observed_panel_size"]
)

# 主ランキング: 観測パネル内で安定して多く確認できるチャンネル。
panel_overlap_df["sample_share_lcb95"] = (
    panel_overlap_df["observed_commenters"]
    .apply(
        lambda n: wilson_lower_bound(
            n,
            N_OBSERVED_PANEL,
        )
    )
)

panel_overlap_df["total_subs"] = pd.to_numeric(
    panel_overlap_df["total_subs"],
    errors="coerce",
)

# 補助ランキング: 小規模チャンネル探索用。
# 比較対象がないため、これを「親和性」として結論づけない。
panel_overlap_df["penetration_per_1M_subs"] = (
    panel_overlap_df["observed_commenters"]
    / panel_overlap_df["total_subs"]
    * 1_000_000
)

panel_overlap_df.loc[
    panel_overlap_df["total_subs"].isna()
    | (panel_overlap_df["total_subs"] <= 0),
    "penetration_per_1M_subs",
] = np.nan

main_columns = [
    "channel_id",
    "channel",
    "observed_commenters",
    "observed_panel_size",
    "sample_share",
    "sample_share_lcb95",
    "total_subs",
    "subscriber_count_hidden",
    "retrieved_at",
]

panel_overlap_df = panel_overlap_df.sort_values(
    [
        "sample_share_lcb95",
        "observed_commenters",
    ],
    ascending=False,
)

print(
    "=== 主ランキング: 観測パネル内での登録割合 "
    "（Wilson下限95%）==="
)
print(
    panel_overlap_df[
        main_columns
    ]
    .head(30)
    .to_string(index=False)
)

print()
print(
    "=== 補助ランキング: 小規模チャンネル探索用 "
    "（主指標ではない）==="
)
print(
    panel_overlap_df[
        panel_overlap_df["penetration_per_1M_subs"].notna()
    ]
    .sort_values(
        [
            "penetration_per_1M_subs",
            "observed_commenters",
        ],
        ascending=False,
    )[
        main_columns
        + ["penetration_per_1M_subs"]
    ]
    .head(30)
    .to_string(index=False)
)

panel_overlap_df.to_csv(
    "channel_panel_overlap_metrics.csv",
    index=False,
)


=== 主ランキング: 観測パネル内での登録割合 （Wilson下限95%）===
              channel_id                                channel  observed_commenters  observed_panel_size  sample_share  sample_share_lcb95  total_subs subscriber_count_hidden                     retrieved_at
UC0PotgTwYxbOZuM9FbWKdcg                     Paranoia_パラノイア【有益】                  470                 1527      0.307793            0.285148      398000                   False 2026-06-21T09:38:52.524786+00:00
UC-N7pA0rR1PrUBOfs2OA0oA          哲理学作家さとうみつろう『神さまとのおしゃべり』チャンネル                  329                 1527      0.215455            0.195561      752000                   False 2026-06-21T09:38:52.524703+00:00
UC67Wr_9pA4I0glIxDt_Cpyw                          両学長 リベラルアーツ大学                  269                 1527      0.176162            0.157874     9820000                   False 2026-06-21T09:38:52.524673+00:00
UC4lN5sizuJraSHqy99xTy6Q                          Naokiman Show                  209                 1527      0.136870   